<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk5_forestdiffusion_autotherm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ForestDiffusion --quiet 2>&1 | tail -20
print("Checking installation...")
import ForestDiffusion
print("✅ ForestDiffusion installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 12.3 MB/s eta 0:00:00
Checking installation...
✅ ForestDiffusion installed


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from ForestDiffusion import ForestDiffusionModel

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import re

print("✅ Imports ready")

✅ Imports ready


In [3]:
!pip install datasets --quiet

In [4]:
print("Loading AutoTherm indoor dataset...")
dataset = load_dataset('kopetri/AutoTherm', 'indoor')
train_df = dataset['train'].to_pandas()

def extract_participant_id(filename):
    match = re.search(r'participant_\d+', filename)
    return match.group() if match else 'unknown'

train_df['participant_id'] = train_df['file_name'].apply(extract_participant_id)

def add_3class(df):
    df = df.copy()
    df['Label_3class'] = df['Label'].apply(
        lambda x: -1 if x <= -2 else (0 if x <= 1 else 1)
    )
    return df

train_df = add_3class(train_df)

test_participants  = ['participant_16', 'participant_14', 'participant_20']
train_participants = [p for p in train_df['participant_id'].unique()
                      if p not in test_participants]

train_split = train_df[train_df['participant_id'].isin(train_participants)]
test_split  = train_df[train_df['participant_id'].isin(test_participants)]

print(f"Train: {len(train_split):,} | Test: {len(test_split):,}")

Loading AutoTherm indoor dataset...


README.md:   0%|          | 0.00/8.57k [00:00<?, ?B/s]

indoor/train-00000-of-00002.parquet:   0%|          | 0.00/29.8M [00:00<?, ?B/s]

indoor/train-00001-of-00002.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

indoor/test-00000-of-00001.parquet:   0%|          | 0.00/7.41M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1566728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

Train: 1,276,709 | Test: 290,019


In [5]:
drop_cols = [
    'file_name', 'Timestamp', 'participant_id',
    'Air-Velocity', 'Metabolic-Rate',
    'Nose', 'Neck', 'RShoulder', 'RElbow',
    'LShoulder', 'LElbow', 'REye', 'LEye', 'REar', 'LEar',
    'Emotion-Self', 'Emotion-ML',
    'Label', 'Label_3class'
]

def prepare_features(df, drop_cols, target_col):
    df = df.copy()
    X = df.drop(columns=drop_cols, errors='ignore')
    y = df[target_col]
    for col in X.select_dtypes(include=['object','category']).columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    return X, y

X_train_7, y_train_7 = prepare_features(train_split, drop_cols, 'Label')
X_test_7,  y_test_7  = prepare_features(test_split,  drop_cols, 'Label')
X_train_3, y_train_3 = prepare_features(train_split, drop_cols, 'Label_3class')
X_test_3,  y_test_3  = prepare_features(test_split,  drop_cols, 'Label_3class')

feature_cols = X_train_7.columns.tolist()

clf_base_7 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_base_7.fit(X_train_7, y_train_7)
base_7_f1 = f1_score(y_test_7, clf_base_7.predict(X_test_7), average='macro')

clf_base_3 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_base_3.fit(X_train_3, y_train_3)
base_3_f1 = f1_score(y_test_3, clf_base_3.predict(X_test_3), average='macro')

print(f"Baseline 7-class: {base_7_f1:.4f} | 3-class: {base_3_f1:.4f}")

Baseline 7-class: 0.2858 | 3-class: 0.7163


In [6]:
# ForestDiffusion needs numeric arrays. We pass the Label column
# separately as the conditioning variable — this is class-conditional
# generation, the mechanism we specifically want to test.

# Stratified 100K sample, same approach as CTGAN/TVAE for fair comparison
sdv_drop_for_sample = [
    'file_name', 'Timestamp', 'participant_id',
    'Air-Velocity', 'Metabolic-Rate',
    'Nose', 'Neck', 'RShoulder', 'RElbow',
    'LShoulder', 'LElbow', 'REye', 'LEye', 'REar', 'LEar',
    'Emotion-Self', 'Emotion-ML', 'Label_3class'
]

fd_train = train_split.drop(columns=sdv_drop_for_sample, errors='ignore').copy()
fd_train['Gender'] = LabelEncoder().fit_transform(fd_train['Gender'].astype(str))

fd_train_sample = fd_train.groupby('Label', group_keys=False).apply(
    lambda x: x.sample(
        n=min(len(x), int(100000 * len(x) / len(fd_train))),
        random_state=42
    )
).reset_index(drop=True)

print(f"Sample size: {len(fd_train_sample):,}")
print(fd_train_sample['Label'].value_counts().sort_index())

# Separate features from label
X_fd = fd_train_sample.drop(columns=['Label']).values.astype(float)
y_fd = fd_train_sample['Label'].values

print(f"\nFitting ForestDiffusion — class-conditional generation...")
print("This may take 5-15 minutes on CPU\n")

forest_model = ForestDiffusionModel(
    X_fd,
    label_y=y_fd,
    n_t=50,
    duplicate_K=1,
    diffusion_type='flow',
    n_jobs=-1
)
print("\n✅ ForestDiffusion fitted!")

Sample size: 99,996
Label
-3     3761
-2     9151
-1    20622
 0    24056
 1    13968
 2    15896
 3    12542
Name: count, dtype: int64

Fitting ForestDiffusion — class-conditional generation...
This may take 5-15 minutes on CPU


✅ ForestDiffusion fitted!


In [9]:
help(forest_model.generate)

Help on method generate in module ForestDiffusion.diffusion_with_trees_class:

generate(batch_size=None, n_t=None, X_covs=None) method of ForestDiffusion.diffusion_with_trees_class.ForestDiffusionModel instance
    # Generate new data by solving the reverse ODE/SDE



In [10]:
print("Generating synthetic data from ForestDiffusion...")

result = forest_model.generate(batch_size=len(fd_train_sample))

print(f"Type of result: {type(result)}")
if isinstance(result, tuple):
    print(f"Number of elements: {len(result)}")
    for i, item in enumerate(result):
        print(f"  Element {i}: shape {getattr(item, 'shape', 'no shape attr')}")
else:
    print(f"Shape: {getattr(result, 'shape', 'no shape attr')}")

Generating synthetic data from ForestDiffusion...
Type of result: <class 'numpy.ndarray'>
Shape: (99996, 19)


In [11]:
print("Generating synthetic data from ForestDiffusion...")

result = forest_model.generate(batch_size=len(fd_train_sample))

print(f"Type: {type(result)}")
print(f"Shape: {result.shape}")
print(f"\nFirst row:")
print(result[0])

Generating synthetic data from ForestDiffusion...
Type: <class 'numpy.ndarray'>
Shape: (99996, 19)

First row:
[ 30.           0.68341982 104.9        195.70545968   0.341
  36.35368314   0.31335273   6.2132393    3.5375213    0.69
  33.6         33.7         36.95        92.4695142    4.41339083
  37.          55.          -0.3561982    3.        ]


In [12]:
print("Generating synthetic data from ForestDiffusion...")

result = forest_model.generate(batch_size=len(fd_train_sample))
print(f"✅ Generated array shape: {result.shape}")

# 19 columns = 18 features + 1 label, label is the LAST column
feature_names = fd_train_sample.drop(columns=['Label']).columns.tolist()
print(f"Number of feature names: {len(feature_names)}")

fd_synthetic = pd.DataFrame(result[:, :-1], columns=feature_names)
fd_synthetic['Label'] = result[:, -1]

# Round and clip the label to valid integer range
fd_synthetic['Label'] = np.round(fd_synthetic['Label']).clip(-3, 3).astype(int)

print("\nLabel distribution — REAL (sample):")
print(fd_train_sample['Label'].value_counts().sort_index())
print("\nLabel distribution — FORESTDIFFUSION SYNTHETIC:")
print(fd_synthetic['Label'].value_counts().sort_index())

print("\nFirst 3 rows of synthetic data:")
print(fd_synthetic.head(3))

Generating synthetic data from ForestDiffusion...
✅ Generated array shape: (99996, 19)
Number of feature names: 18

Label distribution — REAL (sample):
Label
-3     3761
-2     9151
-1    20622
 0    24056
 1    13968
 2    15896
 3    12542
Name: count, dtype: int64

Label distribution — FORESTDIFFUSION SYNTHETIC:
Label
-3     3727
-2     9176
-1    20859
 0    24157
 1    13899
 2    15756
 3    12422
Name: count, dtype: int64

First 3 rows of synthetic data:
         Age  Gender  Weight      Height   Bodyfat   Bodytemp  \
0  21.407267     1.0   104.9  198.000000  0.106030  35.800000   
1  30.000000     0.0    54.2  188.429314  0.217289  36.836045   
2  21.013505     0.0    54.2  155.000000  0.000000  36.525503   

   Sport-Last-Hour  Time-Since-Meal  Tiredness  Clothing-Level  \
0         0.000000         2.894454   2.000000         0.69000   
1         0.000000         1.000000   2.000000         0.62139   
2         0.998977         1.000000   4.283625         0.45000   

   Radia

In [13]:
# Quick manual fidelity check — compare summary statistics
# side by side for the key predictive features

key_features = ['Ambient_Temperature', 'Wrist_Skin_Temperature', 'Radiation-Temp',
                 'PCE-Ambient-Temp', 'Ambient_Humidity', 'GSR', 'Heart_Rate', 'Solar_Radiation']

comparison = pd.DataFrame({
    'Real_mean': fd_train_sample[key_features].mean(),
    'Synthetic_mean': fd_synthetic[key_features].mean(),
    'Real_std': fd_train_sample[key_features].std(),
    'Synthetic_std': fd_synthetic[key_features].std(),
    'Real_min': fd_train_sample[key_features].min(),
    'Synthetic_min': fd_synthetic[key_features].min(),
    'Real_max': fd_train_sample[key_features].max(),
    'Synthetic_max': fd_synthetic[key_features].max(),
}).round(3)

print(comparison)

                        Real_mean  Synthetic_mean  Real_std  Synthetic_std  \
Ambient_Temperature        26.869          27.000     4.540          7.488   
Wrist_Skin_Temperature     33.930          33.355     1.810          3.487   
Radiation-Temp             25.579          25.736     3.917          6.543   
PCE-Ambient-Temp           25.360          25.329     3.702          6.467   
Ambient_Humidity           31.501          32.168     8.833         16.398   
GSR                         0.799           2.014     1.130          2.375   
Heart_Rate                 73.193          91.667    14.313         53.362   
Solar_Radiation             0.286           0.071     0.049          0.455   

                        Real_min  Synthetic_min  Real_max  Synthetic_max  
Ambient_Temperature        17.60          17.60    37.000         37.000  
Wrist_Skin_Temperature     27.91          27.91    36.950         36.950  
Radiation-Temp             16.90          16.90    33.600         33.600

In [14]:
print("Re-fitting ForestDiffusion with n_t=100 (more diffusion steps)...")
print("This may take 10-20 minutes on CPU\n")

forest_model_v2 = ForestDiffusionModel(
    X_fd,
    label_y=y_fd,
    n_t=100,              # doubled from 50
    duplicate_K=1,
    diffusion_type='flow',
    n_jobs=-1
)
print("\n✅ ForestDiffusion v2 fitted!")

Re-fitting ForestDiffusion with n_t=100 (more diffusion steps)...
This may take 10-20 minutes on CPU


✅ ForestDiffusion v2 fitted!


In [15]:
print("Generating synthetic data from ForestDiffusion v2...")

result_v2 = forest_model_v2.generate(batch_size=len(fd_train_sample))
print(f"✅ Generated array shape: {result_v2.shape}")

fd_synthetic_v2 = pd.DataFrame(result_v2[:, :-1], columns=feature_names)
fd_synthetic_v2['Label'] = np.round(result_v2[:, -1]).clip(-3, 3).astype(int)

# Same fidelity check as before — this is the number that matters
comparison_v2 = pd.DataFrame({
    'Real_mean': fd_train_sample[key_features].mean(),
    'Synthetic_v2_mean': fd_synthetic_v2[key_features].mean(),
    'Real_std': fd_train_sample[key_features].std(),
    'Synthetic_v2_std': fd_synthetic_v2[key_features].std(),
}).round(3)

comparison_v2['std_ratio'] = (comparison_v2['Synthetic_v2_std'] / comparison_v2['Real_std']).round(2)

print(comparison_v2)
print("\nLabel distribution — FORESTDIFFUSION v2 SYNTHETIC:")
print(fd_synthetic_v2['Label'].value_counts().sort_index())

Generating synthetic data from ForestDiffusion v2...
✅ Generated array shape: (99996, 19)
                        Real_mean  Synthetic_v2_mean  Real_std  \
Ambient_Temperature        26.869             26.995     4.540   
Wrist_Skin_Temperature     33.930             33.297     1.810   
Radiation-Temp             25.579             25.761     3.917   
PCE-Ambient-Temp           25.360             25.324     3.702   
Ambient_Humidity           31.501             32.378     8.833   
GSR                         0.799              2.007     1.130   
Heart_Rate                 73.193             91.432    14.313   
Solar_Radiation             0.286              0.071     0.049   

                        Synthetic_v2_std  std_ratio  
Ambient_Temperature                7.538       1.66  
Wrist_Skin_Temperature             3.505       1.94  
Radiation-Temp                     6.523       1.67  
PCE-Ambient-Temp                   6.445       1.74  
Ambient_Humidity                  16.506     

In [16]:
print("ForestDiffusion experiment concluded.")
print("Label-conditional generation matched real distribution closely (within 1%).")
print("Feature-level fidelity failed: std ratios 1.65x-9.27x too wide,")
print("unchanged after doubling diffusion steps (n_t=50 -> 100).")
print("Hypothesis: not a step-count problem, likely structural to the")
print("default configuration on this 18-feature correlated dataset.")

ForestDiffusion experiment concluded.
Label-conditional generation matched real distribution closely (within 1%).
Feature-level fidelity failed: std ratios 1.65x-9.27x too wide,
unchanged after doubling diffusion steps (n_t=50 -> 100).
Hypothesis: not a step-count problem, likely structural to the
default configuration on this 18-feature correlated dataset.
